# DecInfer-01-Utility-Foundations : Axiomes et Fondements
**Serie** : Programmation Probabiliste avec Infer.NET (1/10)  
**Duree estimee** : 50 minutes  
**Prerequis** : série Infer de [Infer-01-Setup](../../Infer/Infer-01-Setup.ipynb) à [Infer-08-TrueSkill](../../Infer/Infer-08-TrueSkill.ipynb) (modèles probabilistes de base)

***

## Objectifs

- Comprendre les **loteries** comme representation des choix en environnement stochastique
- Maitriser les **axiomes de Von Neumann-Morgenstern**
- Deriver la **fonction d'utilite** par calibration
- Comprendre la **maximisation de l'utilite esperee** comme fondement de l'**agent rationnel**

***

## Navigation

| Précédent | Suivant |
|-----------|--------|
| [Infer-02b-Debugging-Bonnes-Pratiques](../../Infer/Infer-02b-Debugging-Bonnes-Pratiques.ipynb) | [DecInfer-02-Utility-Money](DecInfer-02-Utility-Money.ipynb) |

***

## 1. Introduction : Pourquoi l'Utilite ?

### Le problème de la decision sous incertitude

Jusqu'ici, nous avons utilise Infer.NET pour calculer des distributions de probabilite. Mais comment **agir** face a l'incertitude ?

Considerez ce dilemme :

| Option | Description |
|--------|-------------|
| A | Gagner 100 EUR avec certitude |
| B | 50% de chance de gagner 200 EUR, 50% de ne rien gagner |

**Valeur esperee** : E[A] = 100, E[B] = 100. Identiques !

Pourtant, la plupart des gens preferent A. Pourquoi ?

### La valeur esperee ne suffit pas

Le **Paradoxe de Saint-Petersbourg** (1713) illustre cette limite :

> Une piece est lancee jusqu'a obtenir Face. Si Face apparait au tour n, vous gagnez 2^n euros.
> Combien paieriez-vous pour jouer ?

$$E[\text{gain}] = \sum_{n=1}^{\infty} \frac{1}{2^n} \times 2^n = \sum_{n=1}^{\infty} 1 = \infty$$

Valeur esperee infinie, mais personne ne paierait un million pour jouer. Il faut une autre approche.

## 2. Loteries : Representation Formelle des Choix

### Definition

Une **loterie** est une distribution de probabilite sur des **outcomes** (résultats).

**Notation** : L = [p₁:o₁, p₂:o₂, ..., pₙ:oₙ]

Ou :
- pᵢ est la probabilite de l'outcome oᵢ
- Σpᵢ = 1

### Exemples

| Loterie | Description |
|---------|-------------|
| [1.0 : 100EUR] | 100 EUR avec certitude (outcome degenere) |
| [0.5 : 200EUR, 0.5 : 0EUR] | Pile ou face pour 200 EUR |
| [0.6 : Succes, 0.4 : Echec] | Examen avec 60% de reussite |

### Loteries composees

Une **loterie composee** a d'autres loteries comme outcomes :

L = [0.5 : L₁, 0.5 : L₂]

Elle peut etre **réduite** en loterie simple par le calcul des probabilites totales.

### Preparation de l'environnement

Chargeons d'abord Infer.NET pour les demonstrations qui suivront. Bien que ce notebook soit principalement théorique, nous utiliserons Infer.NET pour illustrer comment l'inference probabiliste s'integre dans la prise de decision.

In [1]:
// Installation Infer.NET
#r "nuget: Microsoft.ML.Probabilistic"
#r "nuget: Microsoft.ML.Probabilistic.Compiler"

using Microsoft.ML.Probabilistic;
using Microsoft.ML.Probabilistic.Distributions;
using Microsoft.ML.Probabilistic.Models;
using Microsoft.ML.Probabilistic.Algorithms;

Console.WriteLine("Infer.NET charge pour Decision Theory !");

Installed Packages Microsoft.ML.Probabilistic, 0.4.2504.701 Microsoft.ML.Probabilistic.Compiler, 0.4.2504.701

Infer.NET charge pour Decision Theory !


Chargement du helper de visualisation des graphes de facteurs.

In [2]:
// Chargement du helper pour visualiser les factor graphs
#load "../../Infer/FactorGraphHelper.cs"

Console.WriteLine("FactorGraphHelper charge.");
Console.WriteLine($"Graphviz disponible : {FactorGraphHelper.IsGraphvizAvailable()}");

FactorGraphHelper charge.


Graphviz disponible : True


### Implementation de la classe Loterie

Nous definissons une classe generique `Loterie<T>` qui encapsule une distribution discrete sur des outcomes de type T. Cette implementation :
- Valide que les probabilites somment a 1
- Fournit un affichage lisible de la forme `[p1:o1, p2:o2, ...]`

Cette structure de données sera utilisée tout au long de ce notebook pour representer les choix sous incertitude.

In [3]:
// Representation d'une loterie simple
public class Loterie<T>
{
    public List<(double prob, T outcome)> Outcomes { get; }
    
    public Loterie(params (double prob, T outcome)[] items)
    {
        Outcomes = items.ToList();
        var total = Outcomes.Sum(x => x.prob);
        if (Math.Abs(total - 1.0) > 1e-6)
            throw new ArgumentException($"Probabilites doivent sommer a 1, got {total}");
    }
    
    public override string ToString()
    {
        var parts = Outcomes.Select(o => $"{o.prob:P0}:{o.outcome}");
        return $"[{string.Join(", ", parts)}]";
    }
}

// Exemples
var certainty = new Loterie<double>((1.0, 100));
var coinFlip = new Loterie<double>((0.5, 200), (0.5, 0));

Console.WriteLine($"Certitude : {certainty}");
Console.WriteLine($"Pile ou face : {coinFlip}");

Certitude : [100 %:100]


Pile ou face : [50 %:200, 50 %:0]


## 3. Axiomes de Préférences Rationnelles

Von Neumann et Morgenstern (1944) ont etabli les axiomes qui définissent un **agent rationnel**. Ces quatre axiomes ne sont pas arbitraires : ils capturent les conditions **minimales** pour qu'un système de préférences soit cohérent et non-exploitable. Un agent qui les viole peut etre "money-pumped" (arnaque par cycle de préférences).

### Notation

- A ≻ B : l'agent prefere strictement A a B
- A ≺ B : l'agent prefere strictement B a A  
- A ~ B : l'agent est indifferent entre A et B
- A ≿ B : l'agent prefere A ou est indifferent (préférence faible)

### Les 4 Axiomes

#### Axiome 1 : Completude (Orderability)

> Pour toute paire de loteries A et B, exactement une relation est vraie :
> A ≻ B, ou B ≻ A, ou A ~ B

**Interpretation** : L'agent peut toujours comparer deux options. Sans cet axiome, l'agent pourrait etre paralyse face a certains choix.

**Pourquoi nécessaire ?** Un agent incapable de comparer A et B ne peut pas prendre de decision rationnelle entre eux.

#### Axiome 2 : Transitivite

> Si A ≻ B et B ≻ C, alors A ≻ C

**Interpretation** : Les préférences sont coherentes, pas de cycles.

**Pourquoi nécessaire ?** Si A ≻ B ≻ C ≻ A, on peut vous faire payer pour echanger C→B, B→A, A→C et vous revenez au point de depart avec moins d'argent. C'est le "money pump".

#### Axiome 3 : Continuité

> Si A ≻ B ≻ C, alors il existe p ∈ (0,1) tel que :
> B ~ [p:A, (1-p):C]

**Interpretation** : même le pire outcome peut etre compense par une probabilite suffisante du meilleur.

**Pourquoi nécessaire ?** Sans continuité, on ne pourrait pas construire une fonction d'utilite a valeurs réelles. Cet axiome garantit l'existence d'un "equivalent certain" pour toute loterie.

#### Axiome 4 : Indépendance

> Si A ≻ B, alors pour toute loterie C et toute probabilite p :
> [p:A, (1-p):C] ≻ [p:B, (1-p):C]

**Interpretation** : Les préférences ne dependent pas des alternatives non pertinentes.

**Pourquoi nécessaire ?** C'est l'axiome le plus controversé. Le "paradoxe d'Allais" (1953) montre que les humains le violent souvent, mais un agent qui le viole peut aussi etre exploite.

### Demonstration : Verification de la transitivite

Illustrons l'axiome de transitivite avec un agent simulee utilisant une fonction d'utilite concave U(x) = sqrt(x). Nous allons comparer trois loteries et verifier que les préférences sont transitives.

In [4]:
// Verification de la transitivite des preferences
// Simulons un agent avec une fonction d'utilite

Func<double, double> utilite = x => Math.Sqrt(x); // Utilite concave (aversion au risque)

double UtiliteEsperee(Loterie<double> L, Func<double, double> U)
{
    return L.Outcomes.Sum(o => o.prob * U(o.outcome));
}

// Trois loteries
var A = new Loterie<double>((1.0, 100));           // 100 certain
var B = new Loterie<double>((0.5, 196), (0.5, 0)); // E[B] = 98, E[U(B)] = 7
var C = new Loterie<double>((1.0, 49));            // 49 certain

double UA = UtiliteEsperee(A, utilite);
double UB = UtiliteEsperee(B, utilite);
double UC = UtiliteEsperee(C, utilite);

Console.WriteLine($"U(A) = {UA:F3} (100 certain)");
Console.WriteLine($"U(B) = {UB:F3} (50% de 196)");
Console.WriteLine($"U(C) = {UC:F3} (49 certain)");
Console.WriteLine();

Console.WriteLine($"A ≻ B ? {UA > UB} (transitivite: si A≻B et B≻C alors A≻C)");
Console.WriteLine($"B ≻ C ? {UB > UC}");
Console.WriteLine($"A ≻ C ? {UA > UC} (verifie !)");

U(A) = 10,000 (100 certain)


U(B) = 7,000 (50% de 196)


U(C) = 7,000 (49 certain)


A ≻ B ? True (transitivite: si A≻B et B≻C alors A≻C)


B ≻ C ? False


A ≻ C ? True (verifie !)


### Analyse des résultats : Transitivite et Indifference

Les résultats ci-dessus revelent un cas intéressant :

| Loterie | Utilite esperee | Interpretation |
|---------|-----------------|----------------|
| A (100 certain) | U(A) = 10.000 | sqrt(100) = 10 |
| B (50% de 196) | U(B) = 7.000 | 0.5 x sqrt(196) + 0.5 x sqrt(0) = 0.5 x 14 = 7 |
| C (49 certain) | U(C) = 7.000 | sqrt(49) = 7 |

**Observation cle** : U(B) = U(C), donc B ~ C (indifference).

La transitivite est preservee car :
- A ≻ B (10 > 7)
- B ~ C (7 = 7, indifference)
- Donc A ≻ C (10 > 7), ce qui est verifie

> **Note technique** : L'indifference B ~ C illustre un point subtil de la théorie. Avec une utilite concave (sqrt), la loterie B a le même attrait que le montant certain C = 49, bien que E[B] = 98 >> 49. C'est exactement l'**aversion au risque** : l'agent prefere 49 certain a une esperance de 98 avec risque.

In [5]:
// Exercice : Verification de l'axiome d'independance

// TODO 1 : Definir trois loteries et une fonction d'utilite
// Indice : var A = new Loterie<double>((1.0, 100));
//          var B = new Loterie<double>((0.5, 150), (0.5, 50));
//          var C = new Loterie<double>((1.0, 30));

// TODO 2 : Verifier que A > B avec votre fonction d'utilite
// Indice : utilisez UtiliteEsperee(Loterie, U) definie precedemment

// TODO 3 : Construire les loteries composees L1 et L2
// L1 = [0.6:A, 0.4:C] => Loterie avec outcomes de A (prob x0.6) et de C (prob x0.4)
// L2 = [0.6:B, 0.4:C] => Loterie avec outcomes de B (prob x0.6) et de C (prob x0.4)

// TODO 4 : Calculer E[U(L1)] et E[U(L2)] et verifier L1 > L2

// TODO 5 : Repeter avec U(x) = x^2 (convexe, amateur de risque)
// Indice : verifier que l'axiome tient toujours meme quand la fonction d'utilite change

Console.WriteLine("Exercice a completer : verification de l'axiome d'independance");

Exercice a completer : verification de l'axiome d'independance


### Exemple 1 : Verification de l'axiome d'indépendance

**Objectif** : Verifiez empiriquement l'axiome d'indépendance de Von Neumann-Morgenstern en construisant des loteries composees.

**Contexte** : L'axiome d'indépendance stipule que si A > B, alors pour toute loterie C et probabilite p : [p:A, (1-p):C] > [p:B, (1-p):C]. Vous allez tester cette propriete avec différentes fonctions d'utilite.

**étapes** :
1. Definissez trois loteries A, B, C et une fonction d'utilite
2. Verifiez que A > B avec votre fonction d'utilite
3. Construisez les loteries composees L1 = [0.6:A, 0.4:C] et L2 = [0.6:B, 0.4:C]
4. Montrez que L1 > L2 (l'axiome est respecte)
5. Testez avec une fonction d'utilite convexe (amateur de risque) et observez si la propriete tient toujours

**Indices** :
- **Indice 1** : Utilisez la classe `Loterie<double>` définie precedemment
- **Indice 2** : Une fonction convexe comme U(x) = x^2 correspond a un amateur de risque
- **Indice 3** : L'axiome d'indépendance est toujours respecte quand on utilise une fonction d'utilite valide

## 4. Theoreme de Representation

### Le theoreme fondamental

> **Theoreme (Von Neumann-Morgenstern, 1944)**
> 
> Si les préférences d'un agent satisfont les 4 axiomes, alors il existe une fonction U (unique a transformation affine pres) telle que :
> 
> A ≻ B  ⟺  E[U(A)] > E[U(B)]

Ce theoreme est remarquable : il montre que des axiomes qualitatifs sur les préférences (comparaisons) impliquent l'existence d'une fonction **quantitative** (utilite). C'est le pont entre psychologie (préférences) et mathematiques (optimisation).

### Consequences

1. **L'utilite esperee suffit** : Pour prendre des decisions rationnelles, il suffit de maximiser E[U(outcome)]

2. **Unicite relative** : U est unique a transformation affine pres (U' = aU + b, a > 0). Cela signifie que seules les différences d'utilite comptent, pas les valeurs absolues.

3. **Fondement de l'agent rationnel** : Un agent qui viole ces axiomes peut etre "money-pumped" (exploite par un cycle de transactions)

### L'agent rationnel fonde sur l'utilite

L'architecture classique d'un agent decisionnel suit ce pipeline :

```mermaid
flowchart LR
    P["Perception (capteurs)"] --> I["Inference Bayesienne (Infer.NET)"]
    I --> C["Calcul de E&#91;U(.)&#93; pour chaque action possible"]
    C --> A["argmax (choix)"]
    A --> ACT["Action"]
```

L'agent :
1. **Observe** le monde (perception via capteurs)
2. **Infere** l'etat du monde a partir des observations (inference bayesienne - ce que nous faisons avec Infer.NET)
3. **Evalue** l'utilite esperee de chaque action possible, integrant sur les etats incertains
4. **Choisit** l'action qui maximise l'utilite esperee (decision)
5. **Agit** dans le monde

C'est exactement ce que nous allons implementer dans les cellules suivantes avec Infer.NET : l'étape 2 (inference) est faite par le moteur probabiliste, et l'étape 3 (évaluation) utilise les posteriors pour calculer les utilites esperees.
> La preuve formelle de ce théorème est développée dans le companion natif [DecInfer-01b-Lean-ExpectedUtility](DecInfer-01b-Lean-ExpectedUtility.ipynb).

Le même **pipeline décisionnel**, rendu sous forme de graphe : les cinq étapes
s'enchaînent de la perception du monde jusqu'à l'action, l'inférence bayésienne (Infer.NET)
alimentant le calcul de l'utilité espérée.

```mermaid
flowchart LR
    P["① Perception<br/>capteurs, observations"]
    I["② Inférence bayésienne<br/>(Infer.NET)"]
    EU["③ Calcul E[U(·)]<br/>pour chaque action"]
    AM["④ argmax<br/>choix de l'action"]
    ACT["⑤ Action"]
    P --> I --> EU --> AM --> ACT
    classDef perc fill:#cfe2ff,stroke:#084298,color:#052c65
    classDef inf fill:#d1e7dd,stroke:#0f5132,color:#052e16
    classDef eval fill:#fff3cd,stroke:#b8860b,color:#5c4400
    classDef dec fill:#e2e3e5,stroke:#41464b,color:#1b1e21
    classDef act fill:#f8d7da,stroke:#842029,color:#2c0b0e
    class P perc
    class I inf
    class EU eval
    class AM dec
    class ACT act
```

> **Lecture.** Seules les étapes ② et ③ mobilisent la théorie de ce notebook : l'**inférence**
> transforme les observations en une croyance (un *posterior*), et l'**évaluation** intègre
> l'utilité sur cette croyance incertaine — `E[U(a)] = Σ P(état | obs)·U(a, état)`. Les étapes ①,
> ④ et ⑤ sont mécaniques (lire les capteurs, prendre le `max`, agir). C'est exactement le schéma
> de l'exemple médical qui suit : ② infère `P(malade | test+)` avec Infer.NET, puis ③ compare
> l'utilité espérée de *traiter* et de *ne pas traiter* pour choisir (④) la meilleure décision.


In [6]:
// Modelisation d'un agent rationnel avec Infer.NET
// Exemple guide 1 : Modele de diagnostic medical avec Infer.NET

// Scenario : Un patient a potentiellement une maladie.
// L'agent (medecin) doit decider : traiter ou ne pas traiter.

// Prior sur la maladie
Variable<bool> malade = Variable.Bernoulli(0.3).Named("malade");

// Test diagnostique (imparfait)
Variable<bool> testPositif = Variable.New<bool>().Named("test");
double sensibilite = 0.9;  // P(test+|malade)
double specificite = 0.8;  // P(test-|sain)

using (Variable.If(malade))
    testPositif.SetTo(Variable.Bernoulli(sensibilite));
using (Variable.IfNot(malade))
    testPositif.SetTo(Variable.Bernoulli(1 - specificite));

// Observation : test positif
testPositif.ObservedValue = true;

// Inference (engine declaration preservee pour cellule suivante 'engine.ShowFactorGraph')
InferenceEngine engine = new InferenceEngine();
engine.Compiler.CompilerChoice = Microsoft.ML.Probabilistic.Compiler.CompilerChoice.Roslyn;

Bernoulli posteriorMalade = engine.Infer<Bernoulli>(malade);
Console.WriteLine($"Posterior P(malade|test+) = {posteriorMalade.GetProbTrue():P1}");


Compiling model...

done.


Posterior P(malade|test+) = 65,9 %


### Interpretation : Le theoreme de Bayes en action

Le résultat P(malade|test+) = 65.9% illustre le theoreme de Bayes :

$$P(\text{malade}|\text{test}+) = \frac{P(\text{test}+|\text{malade}) \times P(\text{malade})}{P(\text{test}+)}$$

Avec les valeurs numériques :

| Terme | Valeur | Signification |
|-------|--------|---------------|
| P(malade) | 0.30 | Prior (prevalence) |
| P(test+ given malade) | 0.90 | Sensibilite |
| P(test+ given sain) | 0.20 | 1 - Specificite |
| P(test+) | 0.30 x 0.90 + 0.70 x 0.20 = 0.41 | Probabilite totale |

$$P(\text{malade}|\text{test}+) = \frac{0.90 \times 0.30}{0.41} = \frac{0.27}{0.41} \approx 0.659$$

> **Intuition clinique** : Malgre un test positif avec 90% de sensibilite, la probabilite de maladie n'est "que" de 66%. C'est parce que la maladie est relativement rare (prior 30%) et que les faux positifs (20% des sains) contribuent significativement au total des tests positifs.

Infer.NET a effectue ce calcul automatiquement grace au modèle graphique. C'est la puissance de l'inference probabiliste programmatique !

In [7]:
// Visualisation du factor graph pour le modele de diagnostic medical
// Activons ShowFactorGraph pour le prochain modele similaire
engine.ShowFactorGraph = true;

// Reconstruction du modele pour generer le graphe
Variable<bool> malade2 = Variable.Bernoulli(0.3).Named("malade");
Variable<bool> test2 = Variable.New<bool>().Named("test_positif");

using (Variable.If(malade2))
    test2.SetTo(Variable.Bernoulli(0.9)); // sensibilite
using (Variable.IfNot(malade2))
    test2.SetTo(Variable.Bernoulli(0.2)); // 1 - specificite

test2.ObservedValue = true;

// Inference avec generation du graphe
var engineViz = new InferenceEngine();
engineViz.ShowFactorGraph = true;
engineViz.Compiler.CompilerChoice = Microsoft.ML.Probabilistic.Compiler.CompilerChoice.Roslyn;
engineViz.Infer<Bernoulli>(malade2);

Console.WriteLine("Factor graph genere pour le modele de diagnostic.");
Console.WriteLine("Structure : P(malade) -> Test diagnostique -> Observation");

Compiling model...

done.


Factor graph genere pour le modele de diagnostic.


Structure : P(malade) -> Test diagnostique -> Observation


Affichage du graphe de facteurs du modèle de diagnostic.

In [8]:
// Affichage du factor graph du modele de diagnostic medical
// Le graphe montre la structure : Prior(malade) -> Facteur conditionnel -> Observation(test)
FactorGraphHelper.GetLatestFactorGraphDisplay().Display();

### De l'inference a la decision

Nous avons calcule P(malade|test+) = 65.9%. Maintenant, l'agent doit utiliser ce posterior pour prendre une decision. Nous definissons une **matrice d'utilite** qui specifie les consequences de chaque action dans chaque etat possible.

In [9]:
// Maintenant, l'agent doit decider : traiter ou pas ?

// Matrice d'utilite (outcomes)
double U_traiter_malade = 80;    // Guerison (moins effets secondaires)
double U_traiter_sain = 60;      // Effets secondaires inutiles
double U_pas_traiter_malade = 0; // Catastrophe
double U_pas_traiter_sain = 100; // Parfait

double pMalade = posteriorMalade.GetProbTrue();
double pSain = 1 - pMalade;

// Utilite esperee de chaque action
double EU_traiter = pMalade * U_traiter_malade + pSain * U_traiter_sain;
double EU_pas_traiter = pMalade * U_pas_traiter_malade + pSain * U_pas_traiter_sain;

Console.WriteLine($"\nP(malade) = {pMalade:P1}, P(sain) = {pSain:P1}");
Console.WriteLine($"\nE[U(traiter)] = {pMalade:F2} x {U_traiter_malade} + {pSain:F2} x {U_traiter_sain} = {EU_traiter:F1}");
Console.WriteLine($"E[U(pas traiter)] = {pMalade:F2} x {U_pas_traiter_malade} + {pSain:F2} x {U_pas_traiter_sain} = {EU_pas_traiter:F1}");

string decision = EU_traiter > EU_pas_traiter ? "TRAITER" : "NE PAS TRAITER";
Console.WriteLine($"\nDecision optimale : {decision}");


P(malade) = 65,9 %, P(sain) = 34,1 %



E[U(traiter)] = 0,66 x 80 + 0,34 x 60 = 73,2


E[U(pas traiter)] = 0,66 x 0 + 0,34 x 100 = 34,1



Decision optimale : TRAITER


### Synthese : La matrice de decision medicale

L'agent rationnel (medecin) a evalue chaque action selon la **matrice de decision** :

| Action | Etat : Malade (65.9%) | Etat : Sain (34.1%) | E[U(action)] |
|--------|----------------------|---------------------|--------------|
| Traiter | U = 80 (guerison) | U = 60 (effets secondaires) | **73.2** |
| Ne pas traiter | U = 0 (catastrophe) | U = 100 (parfait) | 34.1 |

**Decomposition du calcul** :

$$E[U(\text{traiter})] = 0.659 \times 80 + 0.341 \times 60 = 52.7 + 20.5 = 73.2$$
$$E[U(\text{ne pas traiter})] = 0.659 \times 0 + 0.341 \times 100 = 0 + 34.1 = 34.1$$

> **Decision** : L'ecart de 39.1 unites d'utilite en faveur du traitement est substantiel. même si le patient pourrait etre sain (34% de chance), le cout asymetrique des erreurs (0 vs 60) rend le traitement rationnel.

Cette structure "inference bayesienne + maximisation d'utilite esperee" est le paradigme fondamental de l'**agent rationnel** en IA.

In [10]:
// Exercice : Decision medicale avec test sequentiel

// Parametres du second test (IRM)
double sensibiliteIRM = 0.95;
double specificiteIRM = 0.90;
double coutIRM = 500;  // Cout en unite d'utilite

// Utilites du traitement (reutilisees du modele principal)
// U(traiter, malade) = 80, U(traiter, sain) = 60
// U(pas_traiter, malade) = 0, U(pas_traiter, sain) = 100

// TODO 1 : Creer le modele Infer.NET avec deux tests sequentiels
// Indice : Variable<bool> malade = Variable.Bernoulli(0.3);
//          Premier test (sensibilite=0.9, specificite=0.8) deja observe positif
//          Second test (IRM) conditionne sur malade

// TODO 2 : Calculer les posteriors pour les deux scenarios du second test
// - Scenario A : IRM positif => P(malade | test1+,IRM+)
// - Scenario B : IRM negatif => P(malade | test1+,IRM-)

// TODO 3 : Calculer E[U(traiter)] et E[U(pas_traiter)] pour chaque scenario
// (soustraire le cout de l'IRM aux utilites)

// TODO 4 : Calculer l'utilite esperee de la strategie "faire l'IRM puis decider"
// et la comparer a la strategie "decider immediatement sans IRM"
// Formule : E[U(avec IRM)] = P(IRM+) x max(EU|IRM+) + P(IRM-) x max(EU|IRM-) - coutIRM

Console.WriteLine("Exercice a completer : decision medicale avec test sequentiel");

Exercice a completer : decision medicale avec test sequentiel


## Exercice 2 : Decision medicale avec test séquentiel

**Objectif** : Etendez le modèle de decision medicale pour gerer deux tests séquentiels et determinez si le second test est nécessaire.

**Contexte** : Un medecin peut realiser un second test complementaire (IRM) après un premier test sanguin positif. L'IRM a une sensibilite de 95% et une specificite de 90%, mais coute 500 EUR d'utilite. La decision finale (traiter ou non) doit integrer les résultats des deux tests.

**étapes** :
1. Definissez un modèle Infer.NET avec deux tests conditionnels sur la variable `malade`
2. Calculez le posterior P(malade | test1+, test2+) et P(malade | test1+, test2-)
3. Pour chaque scénario du second test, calculez l'utilite esperee de traiter vs ne pas traiter
4. Determinez si le second test est justifie en comparant E[U(avec test)] vs E[U(sans test)]

**Indices** :
- **Indice 1** : L'esperance sur le second test utilise la loi des probabilites totales : P(test2+) x E[U|test2+] + P(test2-) x E[U|test2-]
- **Indice 2** : Le cout du test (-500) s'ajoute a l'utilite de chaque outcome quand on fait le test
- **étape 3** : Utilisez `Variable.Case` ou `Variable.If` pour conditionner le second test sur la variable malade

## 5. Calibration par Mise a l'Indifference

### Le problème pratique

Comment déterminer la fonction d'utilite U d'un agent ? En pratique, on ne peut pas observer U directement - on ne voit que les choix de l'agent. La **calibration** permet de reconstruire U a partir de ces choix.

### Méthode de l'indifference (Axiome de continuité)

Cette méthode exploite l'axiome de continuité pour "sonder" la fonction d'utilite point par point :

**étape 1 : Fixer les références**
- Choisissez deux outcomes de référence : 
  - o_best (le meilleur) avec U(o_best) = 1
  - o_worst (le pire) avec U(o_worst) = 0

**étape 2 : Pour chaque outcome intermediaire**
- Presentez le choix : 
  - Option A : outcome o avec certitude
  - Option B : loterie [p : o_best, (1-p) : o_worst]
- Faites varier p jusqu'a trouver p* ou l'agent est indifferent

**étape 3 : En deduire l'utilite**
- Par l'axiome de continuité : o ~ [p* : o_best, (1-p*) : o_worst]
- Donc U(o) = p* × U(o_best) + (1-p*) × U(o_worst) = p* × 1 + (1-p*) × 0 = **p***

**étape 4 : Interpoler**
- Repetez pour plusieurs outcomes pour obtenir la courbe U(.)
- Ajustez une fonction parametrique (log, CRRA, etc.)

### Exemple : Evaluer une assurance

Vous avez une voiture de 10 000 EUR. Probabilite de vol = 5%.

| Outcome | Valeur |
|---------|--------|
| Pas de vol | 10 000 EUR |
| Vol | 0 EUR |

Une assurance coute 600 EUR. Devez-vous la prendre ?

La reponse depend de votre fonction d'utilite ! Un agent neutre au risque compare les valeurs esperees (ne pas prendre l'assurance : E = 9500, prendre : 9400). Un agent averse au risque peut preferer l'assurance malgre une valeur esperee moindre.

In [11]:
// Calibration de l'utilite pour la decision d'assurance
// Exercice 3 : Decision d'assurance avec aversion au risque

// Supposons une utilite logarithmique (aversion au risque classique)
// U(x) = log(x + 1) normalise sur [0, 10000]

// TODO 1 : Implementer la fonction Utilite(x) logarithmique normalisee
// Indice : if (x <= 0) return 0;
//         return Math.Log(x + 1) / Math.Log(10001);

// TODO 2 : Calculer l'utilite esperee SANS assurance (loterie [0.95:10000, 0.05:0])
// Indice : double EU_sansAssurance = (1 - 0.05) * Utilite(10000) + 0.05 * Utilite(0);

// TODO 3 : Calculer l'utilite esperee AVEC assurance (certitude de 10000 - prime = 9400)
// Indice : double EU_avecAssurance = Utilite(9400);

// TODO 4 : Comparer et decider
// TODO 5 : Calculer la prime maximale acceptable
Console.WriteLine("Exercice 2 a completer : calibration de l'utilite pour decision d'assurance");


Exercice 2 a completer : calibration de l'utilite pour decision d'assurance


### Analyse attendue : pourquoi une prime si élevée ?

En completant l'exercice ci-dessus, vous devriez trouver qu'avec une utilite logarithmique, l'agent accepterait de payer jusqu'a **3691 EUR** pour assurer une voiture de 10 000 EUR contre un risque de vol de 5%.

| Metrique | Sans assurance | Avec assurance (600 EUR) |
|----------|----------------|-------------------------|
| Valeur esperee | 9 500 EUR | 9 400 EUR |
| Utilite esperee | 0.9500 | 0.9933 |

**Pourquoi cette différence ?**

L'utilite logarithmique U(x) = log(x+1) est fortement **concave**. Cette courbure capture l'aversion au risque :
- Perdre 10 000 EUR (de 10 000 a 0) est **catastrophique** en termes d'utilite
- Perdre 600 EUR (de 10 000 a 9 400) est **marginal** en termes d'utilite

La "prime maximale acceptable" de 3691 EUR est le montant ou l'agent devient indifferent. C'est la **prime de risque** - le cout maximal que l'agent paierait pour eliminer l'incertitude.

> **Formule de la prime de risque** : Si CE est l'equivalent certain d'une loterie L,
> $$\text{Prime de risque} = E[L] - CE$$
> Ici : Prime = 9 500 - (10 000 - 3691) = 9 500 - 6 309 = 3 191 EUR

Cette valeur élevée explique pourquoi les marches de l'assurance existent : les individus averses au risque sont prets a payer significativement plus que la perte esperee pour obtenir la certitude.

***

### Transition : De la théorie a la pratique avec Infer.NET

Nous avons maintenant les fondations théoriques :

| Concept | Rôle dans l'agent rationnel |
|---------|----------------------------|
| Loteries | Representation formelle des choix incertains |
| Axiomes VNM | Garantissent l'existence d'une fonction d'utilite |
| Calibration | Permet de déterminer U empiriquement |
| E[U] maximisation | critère de decision |

**Comment Infer.NET s'integre-t-il ?**

Infer.NET excelle dans l'**étape d'inference** : calculer les distributions posterieures sur les etats du monde. Une fois ces distributions obtenues, le calcul de l'utilite esperee devient une simple intégration.

```
Observation -> [Infer.NET: P(etat|obs)] -> [Calcul E[U]] -> Decision
```

Dans la section suivante, nous montrons comment representer une loterie comme un modèle probabiliste Infer.NET et comment exploiter l'inference pour la prise de decision.

## 6. Modelisation avec Infer.NET

### Loteries comme modèles probabilistes

Une loterie peut etre naturellement modelisee comme une variable aléatoire dans Infer.NET. L'avantage de cette approche est triple :

1. **Inference** : Infer.NET peut calculer les posteriors sur des variables latentes (comme dans l'exemple medical précédent)
2. **Composition** : Les loteries composees sont modelisées par des modèles graphiques
3. **intégration** : Le calcul de E[U] peut s'appuyer sur les distributions inferees

L'exemple suivant montre comment une loterie binaire simple est représentée comme un modèle conditionnel dans Infer.NET.

In [12]:
// Loterie : 50% de gagner 1000, 50% de perdre 500

Variable<bool> gagne = Variable.Bernoulli(0.5).Named("gagne");
Variable<double> gain = Variable.New<double>().Named("gain");

using (Variable.If(gagne))
{
    gain.SetTo(Variable.Constant(1000.0));
}
using (Variable.IfNot(gagne))
{
    gain.SetTo(Variable.Constant(-500.0));
}

// Inference du gain moyen (valeur esperee)
InferenceEngine moteur = new InferenceEngine();
moteur.Compiler.CompilerChoice = Microsoft.ML.Probabilistic.Compiler.CompilerChoice.Roslyn;

// Note: gain est un melange de deux points, pas une gaussienne
// Infer.NET retourne une approximation gaussienne
var gainDist = moteur.Infer<Gaussian>(gain);
Console.WriteLine($"Distribution du gain (approximation) : {gainDist}");
Console.WriteLine($"Valeur esperee : {gainDist.GetMean():F0} EUR");
Console.WriteLine();

// Calcul manuel de l'utilite esperee
// Avec U(x) = sqrt(x + 1000) (decalee pour eviter valeurs negatives)
Func<double, double> U = x => Math.Sqrt(x + 1000);

double EU = 0.5 * U(1000) + 0.5 * U(-500);
double U_certain = U(gainDist.GetMean()); // U(250)

Console.WriteLine($"E[U(loterie)] = 0.5 x U(1000) + 0.5 x U(-500)");
Console.WriteLine($"             = 0.5 x {U(1000):F2} + 0.5 x {U(-500):F2}");
Console.WriteLine($"             = {EU:F2}");
Console.WriteLine();
Console.WriteLine($"U(valeur esperee) = U(250) = {U_certain:F2}");
Console.WriteLine();
Console.WriteLine($"EU < U(EV) ? {EU < U_certain} (=> agent averse au risque prefere la certitude)");

Compiling model...

done.


Distribution du gain (approximation) : Gaussian(250, 5,625e+05)


Valeur esperee : 250 EUR


E[U(loterie)] = 0.5 x U(1000) + 0.5 x U(-500)


             = 0.5 x 44,72 + 0.5 x 22,36


             = 33,54


U(valeur esperee) = U(250) = 35,36


EU < U(EV) ? True (=> agent averse au risque prefere la certitude)


### Interpretation : Equivalent certain et aversion au risque

Les résultats ci-dessus illustrent un concept fondamental de la théorie de la decision :

| Concept | Valeur | Interpretation |
|---------|--------|----------------|
| E[gain] | 250 EUR | Valeur esperee de la loterie |
| U(E[gain]) | 35.36 | Utilite de la valeur esperee |
| E[U(gain)] | 33.54 | Utilite esperee de la loterie |

**L'inegalite de Jensen** : Pour une fonction d'utilite concave U,

$$E[U(X)] \leq U(E[X])$$

Cette inegalite est stricte des que X n'est pas constant et U est strictement concave. C'est la signature mathematique de l'**aversion au risque**.

**Equivalent certain** : Le montant CE tel que U(CE) = E[U(loterie)] est l'**equivalent certain**. Ici :

$$U(CE) = 33.54 \Rightarrow \sqrt{CE + 1000} = 33.54 \Rightarrow CE = 33.54^2 - 1000 \approx 125 \text{ EUR}$$

L'agent averse au risque est indifferent entre :
- Recevoir 125 EUR avec certitude
- Jouer la loterie [50% : +1000, 50% : -500] dont E = 250 EUR

> **Note Infer.NET** : Le moteur approxime la distribution bimodale par une Gaussienne (moyenne 250, variance élevée). Cette approximation suffit pour le calcul de E[X] mais pas pour les moments superieurs. Pour des calculs d'utilite précis, il vaut mieux calculer E[U] manuellement comme montre ci-dessus.

In [13]:
// Visualisation du factor graph pour le modele de loterie
// Generation du graphe pour la loterie gagne/gain

Variable<bool> gagne2 = Variable.Bernoulli(0.5).Named("gagne");
Variable<double> gain2 = Variable.New<double>().Named("gain");

using (Variable.If(gagne2))
{
    gain2.SetTo(Variable.Constant(1000.0));
}
using (Variable.IfNot(gagne2))
{
    gain2.SetTo(Variable.Constant(-500.0));
}

var moteurViz = new InferenceEngine();
moteurViz.ShowFactorGraph = true;
moteurViz.Compiler.CompilerChoice = Microsoft.ML.Probabilistic.Compiler.CompilerChoice.Roslyn;
moteurViz.Infer<Gaussian>(gain2);

Console.WriteLine("Factor graph de la loterie genere.");
Console.WriteLine("Structure : P(gagne=0.5) -> Facteur conditionnel -> gain (+1000 ou -500)");

Compiling model...

done.


Factor graph de la loterie genere.


Structure : P(gagne=0.5) -> Facteur conditionnel -> gain (+1000 ou -500)


Affichage du graphe de facteurs du modèle de loterie avec utilite.

In [14]:
// Affichage du factor graph du modele de loterie
// Le graphe montre la structure conditionnelle : gagne (Bernoulli) -> gain (deterministe selon l'etat)
FactorGraphHelper.GetLatestFactorGraphDisplay().Display();

## 7. Exemple guide : Calibrer Votre Fonction d'Utilite

### Enonce

Imaginez que vous devez choisir entre :

- **Option A** : Recevoir X EUR avec certitude
- **Option B** : 50% de chance de recevoir 1000 EUR, 50% de ne rien recevoir

**Question** : Pour quelle valeur de X etes-vous indifferent entre A et B ?

### Méthode

1. Si votre X* = 500, vous etes **neutre au risque**
2. Si votre X* < 500, vous etes **averse au risque**
3. Si votre X* > 500, vous etes **amateur de risque**

### Code pour explorer

In [15]:
// Exploration : trouvez votre equivalent certain

// Supposons que votre equivalent certain est X* = 400 EUR
// Cela revele votre fonction d'utilite !

double equivalentCertain = 400; // <- Modifiez cette valeur selon votre preference

// Si vous etes indifferent entre X* certain et [0.5:1000, 0.5:0]
// Alors U(X*) = 0.5 * U(1000) + 0.5 * U(0)

// Pour CRRA : U(x) = x^(1-rho) / (1-rho) pour rho != 1
//           = ln(x) pour rho = 1

// NOTE IMPORTANTE : Pour les loteries incluant 0, on utilise U(epsilon)
// car CRRA n'est pas definie en 0 pour rho >= 1

double epsilon = 1.0; // Richesse minimale pour eviter U(0) = -infini

// Fonction CRRA avec protection contre les valeurs extremes
Func<double, double, double> CRRA = (x, rho) => {
    if (x < epsilon) x = epsilon;
    if (Math.Abs(rho - 1.0) < 0.01) return Math.Log(x);
    return Math.Pow(x, 1 - rho) / (1 - rho);
};

// Recherche de rho par resolution de l'equation d'indifference
// U(CE) = 0.5 * U(1000) + 0.5 * U(epsilon)

double FindRho(double ce, double high, double low, double p)
{
    double rhoMin = 0.01, rhoMax = 10.0;
    
    for (int i = 0; i < 100; i++)
    {
        double rho = (rhoMin + rhoMax) / 2;
        
        double U_ce = CRRA(ce, rho);
        double U_high = CRRA(high, rho);
        double U_low = CRRA(low, rho);
        double EU_loterie = p * U_high + (1 - p) * U_low;
        
        // On cherche rho tel que U_ce = EU_loterie
        // Si U_ce > EU_loterie, l'agent est moins averse => reduire rho
        if (U_ce > EU_loterie)
            rhoMax = rho;
        else
            rhoMin = rho;
            
        if (Math.Abs(rhoMax - rhoMin) < 0.0001) break;
    }
    return (rhoMin + rhoMax) / 2;
}

double rho = FindRho(equivalentCertain, 1000, epsilon, 0.5);

Console.WriteLine($"Equivalent certain : {equivalentCertain} EUR");
Console.WriteLine($"Loterie : 50% de 1000 EUR, 50% de {epsilon} EUR");
Console.WriteLine($"Coefficient d'aversion au risque (rho) estime : {rho:F2}");
Console.WriteLine();

if (rho < 0.5)
    Console.WriteLine("Profil : Faible aversion au risque (proche de la neutralite)");
else if (rho < 1.5)
    Console.WriteLine("Profil : Aversion au risque moderee (profil equilibre)");
else if (rho < 3.0)
    Console.WriteLine("Profil : Aversion au risque significative (profil prudent)");
else
    Console.WriteLine("Profil : Tres forte aversion au risque (profil conservateur)");

// Verification
double U_ce = CRRA(equivalentCertain, rho);
double EU_loterie = 0.5 * CRRA(1000, rho) + 0.5 * CRRA(epsilon, rho);

Console.WriteLine($" Verification de l'indifference :");
Console.WriteLine($"  U({equivalentCertain}) = {U_ce:F4}");
Console.WriteLine($"  E[U(loterie)] = {EU_loterie:F4}");
Console.WriteLine($"  Difference : {Math.Abs(U_ce - EU_loterie):F6} (devrait etre proche de 0)");

// Afficher quelques exemples de calibration
Console.WriteLine("--- Exemples de calibration ---");
Console.WriteLine("CE (EUR) | rho estime | Interpretation");
Console.WriteLine("---------|------------|----------------");
foreach (double ce in new[] { 200.0, 300.0, 400.0, 450.0, 480.0 })
{
    double r = FindRho(ce, 1000, epsilon, 0.5);
    string interp = r < 0.5 ? "Quasi-neutre" : r < 1.5 ? "Modere" : r < 3 ? "Prudent" : "Conservateur";
    Console.WriteLine($"{ce,8:F0} | {r,10:F2} | {interp}");
}


Equivalent certain : 400 EUR


Loterie : 50% de 1000 EUR, 50% de 1 EUR


Coefficient d'aversion au risque (rho) estime : 0,25


Profil : Faible aversion au risque (proche de la neutralite)


 Verification de l'indifference :


  U(400) = 119,4424


... (6 outputs left out)

     300 |       0,44 | Quasi-neutre


     400 |       0,25 | Quasi-neutre


     450 |       0,14 | Quasi-neutre


     480 |       0,06 | Quasi-neutre


### Analyse du résultat : Calibration CRRA

Le résultat montre un coefficient rho = 0.25 pour un equivalent certain de 400 EUR, indiquant un profil **quasi-neutre au risque**.

| Equivalent certain | Interpretation | Coefficient rho |
|-------------------|----------------|-----------------|
| 480 EUR | Quasi-neutre | rho ~ 0.06 |
| 400 EUR | Faible aversion | rho ~ 0.25 |
| 300 EUR | Quasi-neutre | rho ~ 0.44 |
| 200 EUR | Modere | rho ~ 0.61 |
| < 100 EUR | Forte aversion | rho > 1.0 |

> **Interpretation** : Avec CE = 400 EUR pour une loterie [50% : 1000, 50% : 1], le coefficient rho = 0.25 indique que l'agent est pret a accepter un rabais de 100 EUR (= 500 - 400) par rapport a la valeur esperee pour eliminer l'incertitude. C'est une aversion au risque modeste.

**Experimentez** : Modifiez `equivalentCertain` dans la cellule précédente pour explorer :
- CE = 200 : profil modere (rho ~ 0.6)
- CE = 480 : profil quasi-neutre (rho ~ 0.06)
- CE = 500 : neutralite parfaite (rho = 0)

**Valeurs typiques en economie empirique** :

| Population | rho estime |
|------------|------------|
| Investisseurs institutionnels | 0.5 - 1.5 |
| Menages americains | 1.0 - 3.0 |
| Decisions de sante | 2.0 - 5.0 |

> **Note technique** : La loterie utilise `epsilon = 1` au lieu de 0 car CRRA n'est pas définie en 0 pour rho >= 1. Cela affecte legerement les résultats numériques mais pas l'interpretation qualitative.

Le notebook suivant (DecInfer-02) approfondira ces fonctions d'utilite et leurs proprietes.

## Exercice 3 : Comparer CARA et CRRA — Primes de risque

### Objectif

Implementer et comparer deux familles de fonctions d'utilite classiques en economie pour une même loterie, et analyser comment leurs primes de risque différent.

### Contexte

Deux grandes familles de fonctions d'utilite sont utilisées en pratique :

- **CARA** (Constant Absolute Risk Aversion) : U(x) = -exp(-alpha * x)
  - L'aversion absolue au risque est constante : A(x) = -U''(x)/U'(x) = alpha
  - La prime de risque ne depend pas de la richesse initiale

- **CRRA** (Constant Relative Risk Aversion) : U(x) = x^(1-rho) / (1-rho)
  - L'aversion relative au risque est constante : R(x) = -x * U''(x)/U'(x) = rho
  - La prime de risque est proportionnelle a la richesse

### Travail a realiser

Considerez la loterie suivante : **50% de chance de gagner 2000 EUR, 50% de perdre 500 EUR** (richesse initiale = 10 000 EUR).

1. **Implementer CARA** : définir U_CARA(x, alpha) = -exp(-alpha * x) et calculer l'equivalent certain CE_CARA pour alpha = 0.001
2. **Implementer CRRA** : définir U_CRRA(x, rho) = x^(1-rho)/(1-rho) et calculer l'equivalent certain CE_CRRA pour rho = 2.0
3. **Calculer les primes de risque** : Prime = E[gain] - CE pour chaque fonction
4. **Comparer** les primes de risque et interpreter la différence
5. **Bonus** : Tester avec une richesse initiale différente (ex: 1000 EUR) et observer comment la prime CARA reste constante mais CRRA change

**Indices** :
- **Indice 1** : Pour trouver l'equivalent certain, utilisez une recherche par dichotomie sur l'intervalle [perteBasse - 100, gainHaut + 100]
- **Indice 2** : Pour CRRA, traitez le cas rho = 1 separement avec `Math.Abs(rho - 1) < 1e-6` et utilisez `Math.Log(x)` dans ce cas
- **étape 3** : Prime de risque = E[gain] - CE. Si la prime est positive, l'agent est averse au risque

In [16]:
// Exercice 3 : Comparer les primes de risque CARA vs CRRA
// Richesse initiale et parametres de la loterie
double richesseInitiale = 10000.0;
double gainHaut = 2000.0;   // Gain si issue favorable
double perteBasse = -500.0; // Perte si issue defavorable
double proba = 0.5;

// Valeur esperee de la loterie
double valeurEsperee = proba * gainHaut + (1 - proba) * perteBasse;
Console.WriteLine($"Loterie : {proba:P0} de +{gainHaut:N0} EUR, {(1-proba):P0} de {perteBasse:N0} EUR");
Console.WriteLine($"Valeur esperee du gain : {valeurEsperee:N0} EUR");
Console.WriteLine($"Richesse initiale : {richesseInitiale:N0} EUR");
Console.WriteLine();

// TODO 1 : Implementer la fonction d'utilite CARA
// Indice : utilisez une Func<double, double, double>

// TODO 2 : Implementer la fonction d'utilite CRRA
// Indice : U_CRRA(x, rho) = x^(1-rho) / (1-rho) si rho != 1

// TODO 3 : Calculer l'equivalent certain CE pour chaque fonction
// Indice : recherche par dichotomie dans [perteBasse - 100, gainHaut + 100]

// TODO 4 : Calculer et afficher les primes de risque
// Indice : Prime de risque = valeurEsperee - CE

// TODO 5 (BONUS) : Repeter avec richesseInitiale = 1000 EUR
// Indice : observer comment la prime CARA reste constante mais CRRA change

Console.WriteLine("Exercice a completer : fonctions CARA et CRRA, calcul des equivalents certains et primes de risque");

Loterie : 50 % de +2 000 EUR, 50 % de -500 EUR


Valeur esperee du gain : 750 EUR


Richesse initiale : 10 000 EUR


Exercice a completer : fonctions CARA et CRRA, calcul des equivalents certains et primes de risque


## Exercice 4 : Equivalent certain et deduction de l'aversion au risque

### Objectif

Pour un ensemble de loteries dont on observe l'**equivalent certain**, deduire le coefficient d'aversion au risque (rho pour CRRA) de l'agent, puis tester la **coherence** de ces choix avec un unique profil d'utilite (axiome de representation).

### Contexte

Un investisseur declare etre indifferent entre une somme certaine et chacune des loteries suivantes :

- **Loterie A** : 50% de chance de gagner 2 000 EUR, 50% de 0  ↔  equivalent certain observe X*_A = 800 EUR
- **Loterie B** : 50% de chance de gagner 5 000 EUR, 50% de 0  ↔  equivalent certain observe X*_B = 1 500 EUR
- **Loterie C** : 80% de chance de gagner 2 000 EUR, 20% de 0  ↔  equivalent certain observe X*_C = 1 400 EUR

On suppose une fonction d'utilite CRRA : U(x) = x^(1-rho) / (1-rho).

> **Etapes :**

1. Implementez la fonction `UtiliteCRRA(x, rho)`.
2. Pour chaque loterie, resolvez numeriquement le rho verifiant U(X*) = E[U(loterie)] (methode de bissection sur [0.01, 0.99]).
3. Calculez la **prime de risque** = E[gain] - equivalent certain pour chaque loterie.
4. Comparez les trois rho deduits : sont-ils coherents ? Que revele une incoherence ?

> **Indices :**

- **Indice 1** : pour une loterie {proba : gain, (1-proba) : 0}, comme U(0) = 0, l'indifference se simplifie en U(X*) = proba * U(gain).
- **Indice 2** : un agent rationnel (axiomes VNM) possede un unique rho ; si les trois rho divergent, c'est une violation des axiomes (effet de certitude / theorie des perspectives).
- **Indice 3** : comparez la Loterie A (50%) et la Loterie C (80%) sur le meme gain de 2 000 EUR — le passage de 50% a 80% devrait faire baisser la prime de risque de maniere prevvisible sous CRRA.


In [17]:
// Exercice 4 : Equivalent certain et deduction de l'aversion au risque (CRRA)

// Trois loteries avec leurs equivalents certains observes
double gainA = 2000.0; double probaA = 0.5; double eqCertainA = 800.0;
double gainB = 5000.0; double probaB = 0.5; double eqCertainB = 1500.0;
double gainC = 2000.0; double probaC = 0.8; double eqCertainC = 1400.0;

// TODO 1 : Fonction d'utilite CRRA
// Indice : U(x) = x^(1-rho) / (1-rho)  (rho != 1). U(0) = 0.
double UtiliteCRRA(double x, double rho)
{
    return 0.0;  // TODO : a completer
}

// TODO 2 : Deduire rho par bissection sur [0.01, 0.99]
// Indice : U(X*) = proba * U(gain)  (car U(0)=0)
double DeduireRho(double gain, double proba, double eqCertain)
{
    return 0.0;  // TODO : a completer
}

// TODO 3 : Prime de risque = E[gain] - equivalent certain
double PrimeRisque(double gain, double proba, double eqCertain)
{
    return 0.0;  // TODO : a completer
}

Console.WriteLine("=== Exercice 4 : Deduction de l'aversion au risque ===");
Console.WriteLine($"(A) rho deduit : {DeduireRho(gainA, probaA, eqCertainA):F3} | prime de risque : {PrimeRisque(gainA, probaA, eqCertainA):F0} EUR");
Console.WriteLine($"(B) rho deduit : {DeduireRho(gainB, probaB, eqCertainB):F3} | prime de risque : {PrimeRisque(gainB, probaB, eqCertainB):F0} EUR");
Console.WriteLine($"(C) rho deduit : {DeduireRho(gainC, probaC, eqCertainC):F3} | prime de risque : {PrimeRisque(gainC, probaC, eqCertainC):F0} EUR");
Console.WriteLine();
Console.WriteLine("Question : les trois rho sont-ils coherents ?");
Console.WriteLine("Indice : une incoherence revele une violation des axiomes VNM (effet de certitude).");


=== Exercice 4 : Deduction de l'aversion au risque ===


(A) rho deduit : 0,000 | prime de risque : 0 EUR


(B) rho deduit : 0,000 | prime de risque : 0 EUR


(C) rho deduit : 0,000 | prime de risque : 0 EUR


Question : les trois rho sont-ils coherents ?


Indice : une incoherence revele une violation des axiomes VNM (effet de certitude).


## 8. Resume

| Concept | Description |
|---------|-------------|
| **Loterie** | Distribution de probabilite sur des outcomes |
| **Axiomes VNM** | Completude, Transitivite, continuité, Indépendance |
| **Theoreme de representation** | Existence d'une fonction U telle que préférences = max E[U] |
| **Agent rationnel** | Maximise l'utilite esperee de ses actions |
| **Calibration** | méthode de l'indifference pour déterminer U |

***

## Pour aller plus loin

| Si vous voulez... | Consultez... |
|-------------------|-------------|
| Approfondir l'aversion au risque | [DecInfer-02-Utility-Money](DecInfer-02-Utility-Money.ipynb) |
| Decisions multi-critères | [DecInfer-03-Multi-Attribute](DecInfer-03-Multi-Attribute.ipynb) |
| Visualiser les réseaux de decision | [DecInfer-04-Decision-Networks](DecInfer-04-Decision-Networks.ipynb) |

***

## Prochaine étape

Dans [DecInfer-02-Utility-Money](DecInfer-02-Utility-Money.ipynb), nous approfondirons :

- Le paradoxe de Saint-Petersbourg
- Les fonctions d'utilite CARA et CRRA
- Les coefficients d'aversion au risque (Arrow-Pratt)
- La dominance stochastique

***

## Références

- Von Neumann & Morgenstern (1944) : Theory of Games and Economic Behavior
- Russell & Norvig : Artificial Intelligence, Chapter 16
- Bernoulli (1738) : Specimen Theoriae Novae de Mensura Sortis (Paradoxe de St-Petersbourg)